In [14]:
import os
import pickle
import warnings
from dotenv import load_dotenv

# Suppress annoying deprecation messages
warnings.filterwarnings("ignore", category=DeprecationWarning)

# 1. Core Vector & Embedding Components
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# 2. Correctly separated retriever paths
from langchain_community.retrievers import BM25Retriever  # Keyword search
from langchain_classic.retrievers import EnsembleRetriever     # Hybrid Blending Engine

# 3. LLM Orchestration
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

In [15]:
load_dotenv()

CHROMA_DIR = "chroma_db"
BM25_PKL_PATH = os.path.join(CHROMA_DIR, "bm25_corpus.pkl")

In [16]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7788.94it/s]


In [17]:
vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)
chroma_retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

C:\Users\prade\AppData\Local\Temp\ipykernel_2152\3588483913.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=CHROMA_DIR, embedding_function=embeddings)


In [18]:
if not os.path.exists(BM25_PKL_PATH):
    raise FileNotFoundError("⚠️ Could not find bm25_corpus.pkl. Please run createDB.py first!")

with open(BM25_PKL_PATH, "rb") as f:
    bm25_corpus_data = pickle.load(f)

In [19]:
from langchain_core.documents import Document
bm25_docs = [
    Document(page_content=item["page_content"], metadata=item["metadata"]) 
    for item in bm25_corpus_data
]
bm25_retriever = BM25Retriever.from_documents(bm25_docs)
bm25_retriever.k = 3

In [20]:
hybrid_retriever = EnsembleRetriever(
    retrievers=[chroma_retriever, bm25_retriever],
    weights=[0.6, 0.4]
)

In [21]:
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0.4,
    max_tokens=500
)

In [22]:
system_prompt = (
    "You are an elite, professional, and friendly AI assistant representing Pradeep Kumar Singh.\n"
    "Your job is to answer portfolio visitors' questions accurately using only the provided context below.\n"
    "Keep responses concise, clear, and highly professional.\n"
    "If someone asks for contact details, projects, or strengths, use the context directly.\n"
    "If you do not know the answer based on the context, politely say you don't have that information.\n\n"
    "Context:\n{context}"
)

In [23]:
prompt_template = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

In [24]:
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# 8. Manage Memory (Chat History)
message_history_store = {}

In [25]:
def get_session_history(session_id: str):
    if session_id not in message_history_store:
        message_history_store[session_id] = ChatMessageHistory()
    return message_history_store[session_id]

In [26]:
def ask_personal_bot(user_query: str, session_id: str = "portfolio_user"):
    # Retrieve contextual documents using Hybrid Search
    retrieved_documents = hybrid_retriever.invoke(user_query)
    context_str = format_docs(retrieved_documents)
    
    # Formulate the chain pipeline
    brain_chain = prompt_template | llm
    
    conversational_chain = RunnableWithMessageHistory(
        brain_chain,
        get_session_history,
        input_messages_key="input",
        history_messages_key="history"
    )
    
    # Run chain and pass variable values
    response = conversational_chain.invoke(
        {"input": user_query, "context": context_str},
        config={"configurable": {"session_id": session_id}}
    )
    
    return response.content

In [28]:
# --- TEST EXAMPLES ---
if __name__ == "__main__":
    print("🤖 Bot Ready for Testing!")
    # Test keyword extraction (e.g. checking specific projects inside projects.json
    # Test follow-up context memory
    q1= "why should we hire pradeep?"
    print(f"\nUser: {q1}\nBot: {ask_personal_bot(q1)}")

🤖 Bot Ready for Testing!


C:\Users\prade\AppData\Local\Temp\ipykernel_2152\1655287823.py:7: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  print(f"\nUser: {q1}\nBot: {ask_personal_bot(q1)}")



User: why should we hire pradeep?
Bot: I should be hired because I bring a unique combination of Artificial Intelligence and Full Stack Development skills, practical project experience, a strong learning mindset, and the ability to build complete solutions from idea to deployment.

As a skilled AI Engineer and Full Stack Developer, I can contribute to a forward-thinking organization by:

- Designing and building intelligent AI-powered applications that solve real-world problems
- Developing scalable and efficient backend services using Node.js, Express.js, and MongoDB
- Creating responsive and interactive user interfaces using React, Tailwind CSS, and JavaScript
- Analyzing and visualizing complex data using Python, Pandas, NumPy, and Matplotlib
- Implementing machine learning models using Scikit-learn, TensorFlow, and Keras
- Collaborating with cross-functional teams to deliver high-quality software products

My strengths include:

- Strong problem-solving skills and attention to det